# Check evidence packets

For the two reviewers. Change `packet_id`, then restart the kernel and run all cells.
Run the setup cell first; later cells use its variables.
Nothing here changes files or calls a model.

Sources are extracted from PDFs in `private/pdfs/` into `.txt` and `.json` files in `private/sources/`. The JSON records the PDF filename, hashes, extraction details, and page spans. Apple uses `SRC_01`; Microsoft uses `SRC_02`.

In [ ]:
import json
from pathlib import Path
from decimal import Decimal
from pprint import pprint

# Find the repo even when Jupyter starts in src/ or a notebook folder.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/benchmark/private/packets").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from inside the agent_test repository.")
folder = repo / "data/benchmark"
packet_id = "EP_0001"  # EP_0001 = Apple, EP_0002 = Microsoft

packet = json.loads((folder / f"private/packets/{packet_id}.json").read_text(encoding="utf-8"))
masked_packet = json.loads((folder / f"public/evidence_packets/{packet_id}.json").read_text(encoding="utf-8"))
source_id = packet["source_id"]
source = (folder / f"private/sources/{source_id}.txt").read_text(encoding="utf-8")
source_info = json.loads((folder / f"private/sources/{source_id}.json").read_text(encoding="utf-8"))

print("Company:", packet["company"]["name"])
print("Original source:", source_info.get("input_file") or source_info.get("source_url"))
print("Saved text:", folder / f"private/sources/{source_id}.txt")

## The evidence packet

`private` has the original names and source references. `public` has the masked version the agents will see.

In [ ]:
pprint(packet, sort_dicts=False)

## Source text vs stored evidence

`source_span` is `[start, end]` in the saved source text. The end character is excluded.
Each comparison should print `Equal: True`. Use the printed locator to check the original filing too.

In [ ]:
for evidence in packet["evidence"]:
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]

    print("\n---", evidence["evidence_id"], "---")
    print("Location:", evidence["locator"])
    print("Source span:", [start, end])
    print("\nFROM SOURCE:\n", source_excerpt)
    print("\nIN PACKET:\n", evidence["text"])
    print("\nEqual:", source_excerpt == evidence["text"])

## Source numbers vs stored numbers

Find each recorded source number in the original excerpt and compare it with the stored value.
Also check the printed context: a matching number alone does not prove it belongs to the right row, year, or unit.

In [ ]:
for fact in packet["facts"]:
    evidence = next(e for e in packet["evidence"] if e["evidence_id"] == fact["evidence_id"])
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]
    position = source_excerpt.find(fact["source_value"])
    source_number = source_excerpt[position:position + len(fact["source_value"])] if position >= 0 else None

    print("\n---", fact["fact_id"], fact["metric"], "---")
    print("Evidence:", fact["evidence_id"], "|", fact["locator"])
    print("Period:", fact["period"], "| Unit:", fact["unit"])
    if position >= 0:
        print("Source context:\n", source_excerpt[max(0, position - 100):position + len(fact["source_value"]) + 100])
    print("Source number:", source_number)
    print("Packet number:", fact["value"])
    equal = source_number is not None and Decimal(source_number.replace(",", "")) == Decimal(fact["value"])
    print("Equal:", equal)

## Original vs masked

Read each pair. Names and brands should change; the numbers and meaning should stay the same.

In [ ]:
for evidence in packet["evidence"]:
    masked = next(e for e in masked_packet["evidence"] if e["evidence_id"] == evidence["evidence_id"])
    print("\n---", evidence["evidence_id"], "---")
    print("ORIGINAL:\n", evidence["text"])
    print("\nMASKED:\n", masked["text"])

print("\nStored numbers unchanged by masking:")
for fact in packet["facts"]:
    masked_fact = next(f for f in masked_packet["facts"] if f["fact_id"] == fact["fact_id"])
    print(fact["fact_id"], fact["value"], "->", masked_fact["value"], "Equal:", fact["value"] == masked_fact["value"])

## Review notes

Edit this cell after checking the PDF, numbers, and masking.

- Packet reviewed: pending
- reviewer_1: pending; notes:
- reviewer_2: pending; notes:

Save a copy of this notebook in `private/reviews/` with the packet ID in its filename
to keep the notes. These notes review the evidence, not an Analyst report.